# SourceStretching — the M1 induction source term $S=(b\cdot\nabla)u$ (2D + 3D)

`MagneticStandalone` · ASN 28 validation case (TRUST-LiMe).


## Introduction

Validation of the **standalone magnetic (induction) equation** added by
TRUST-LiMe:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

with the magnetic diffusivity $\eta$ and the transporting velocity $u$ supplied
as data (no Navier–Stokes coupling in this first phase). Every case is
**self-contained** (inline `Mailler`/`Pave` mesh, no `.med` file) and records
point probes (`Postraitement/Sondes`) into `.son` files which are compared
below against a closed-form solution.

This notebook runs the 2D and 3D versions of this test case: ``source_stretching``. Per the TRUST
validation-form layout (`Validation/Rapports_automatiques/<Category>/<FormName>/`),
the other `MagneticStandalone` cases live in their own sibling folders, each
with `<FormName>/<FormName>.ipynb` + `src/<case>.data`.


Both dimensions are run from a single shared data template in this form; the 3D mesh is converted with `Polyedriser dom`.
## What it validates

The `sources { induction_stretching { } }` block adds $S=(b\cdot\nabla)u$ to
the right-hand side. The module transports a single scalar component, so with
a uniform $b=B_0$ and a **divergence-free** velocity $u=(x,-y)$ (so
$\nabla u=\mathrm{diag}(1,-1)$), convection and diffusion vanish identically
and

$$\frac{d b}{dt}=B_0\frac{\partial u_0}{\partial x}=B_0,$$

i.e. $b(t)=B_0e^{t}$. A wrong factor, a transposed gradient index, or a
dropped term changes the exponential rate, so this is a sharp absolute check
of the source magnitude and sign.

## Contracts checked

| # | Modelling decision | Check |
|---|---|:--:|
| — | M1 stretching source $S=(b\cdot\nabla)u$ | $b\sim e^{t}$ |


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.


In [ ]:
import numpy as np
import os
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, name, subdir=None):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    if subdir is None:
        subdir = "3D/coarse" if "_3d" in case.lower() else "coarse"
    path = f"{BUILD}/{subdir}/{case}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def ratio_error(case, tags, ref):
    """Max |b(t)/b(0) - ref(t)| over the given probes and all recorded times."""
    worst = 0.0
    for tag in tags:
        t, b = read_son(case, tag)
        if b[0] == 0:
            continue
        worst = max(worst, float(np.max(np.abs(b / b[0] - ref(t)))))
    return worst

print("BUILD =", BUILD)


## Run both dimensions

Runs on **2 processors** (`nbProcs=2` + `case.partition()`). The dataset's
commented `# BEGIN PARTITION` / `# BEGIN SCATTER` blocks let `trust -partition`
build the parallel `PAR_` variant (upstream `grad_rot.data` pattern).


In [ ]:
from string import Template
from trustutils import run

NB_PROCS = 2
run.reset()

# Mesh geometries and physics/post-processing blocks are the two dimensions
# of the same src template. The 3D case uses a distinct target data name
# and build subtree, so partition zones and probe files cannot collide.
MESH_BLOCKS = {
    2: r"""Mailler dom
{
    Pave pavé
    {
        Origine              0.  0.
        Nombre_de_Noeuds     $nx $ny
        Longueurs            1.  1.
    }
    {
        Bord gauche   X = 0.    0. <= Y <= 1.
        Bord droite   X = 1.    0. <= Y <= 1.
        Bord bas      Y = 0.    0. <= X <= 1.
        Bord haut     Y = 1.    0. <= X <= 1.
    }
}""",
    3: r"""Mailler dom
{
    Pave cube
    {
        Origine              0.  0.  0.
        Nombre_de_Noeuds     $nx $ny $nz
        Longueurs            1.  1.  1.
    }
    {
        Bord gauche   X = 0.    0. <= Y <= 1.  0. <= Z <= 1.
        Bord droite   X = 1.    0. <= Y <= 1.  0. <= Z <= 1.
        Bord bas      Y = 0.    0. <= X <= 1.  0. <= Z <= 1.
        Bord haut     Y = 1.    0. <= X <= 1.  0. <= Z <= 1.
        Bord devant   Z = 0.    0. <= X <= 1.  0. <= Y <= 1.
        Bord derriere Z = 1.    0. <= X <= 1.  0. <= Y <= 1.
    }
}

Polyedriser dom""",
}
DIMENSION_BLOCKS = {
    2: {
        'boundary_block': r"""boundary_conditions
        {
            # b is uniform, so zero-flux everywhere is consistent #
            gauche  Neumann_Paroi_adiabatique
            droite  Neumann_Paroi_adiabatique
            bas     Neumann_Paroi_adiabatique
            haut    Neumann_Paroi_adiabatique
        }""",
        'post_block': r"""Postraitement
  {
    Sondes
    {
      # the transported scalar field must follow exp(t) #
      sonde_b0 grav magnetic_field Periode 1e-6 Points 1 0.5 0.5
    }
  }""",
        'velocity_line': r"""        vitesse_magnetique     Champ_Fonc_xyz dom 2 x -y""",
    },
    3: {
        'boundary_block': r"""boundary_conditions
        {
            # b is uniform, so zero-flux everywhere is consistent #
            gauche   Neumann_Paroi_adiabatique
            droite   Neumann_Paroi_adiabatique
            bas      Neumann_Paroi_adiabatique
            haut     Neumann_Paroi_adiabatique
            devant   Neumann_Paroi_adiabatique
            derriere Neumann_Paroi_adiabatique
        }""",
        'post_block': r"""Postraitement
  {
    Sondes
    {
      # the transported scalar field must follow exp(t) #
      sonde_b0 grav magnetic_field Periode 1e-6 Points 1 0.5 0.5 0.5
    }
  }""",
        'velocity_line': r"""        vitesse_magnetique     Champ_Fonc_xyz dom 3 x -y 0.""",
    },
}

MESHES_2D = {'coarse': {'nx': 17, 'ny': 17}, 'standard': {'nx': 33, 'ny': 33}, 'fine': {'nx': 65, 'ny': 65}}
MESHES_3D = {'coarse': {'nx': 17, 'ny': 17, 'nz': 17}, 'standard': {'nx': 33, 'ny': 33, 'nz': 33}, 'fine': {'nx': 65, 'ny': 65, 'nz': 65}}

def register_dimension(dim, meshes, prefix="", target_data=None, nb_procs=NB_PROCS, partition=True):
    registered = {}
    for level, mesh in meshes.items():
        dic = {"dim": dim,
               "mesh_block": Template(MESH_BLOCKS[dim]).substitute(**mesh),
               **DIMENSION_BLOCKS[dim]}
        target = f"{prefix}/{level}" if prefix else level
        kwargs = {"nbProcs": nb_procs, "excluNR": True}
        if target_data is not None:
            kwargs["targetData"] = target_data
        registered[level] = run.addCaseFromTemplate('source_stretching.data', target, dic=dic, **kwargs)
        if partition:
            registered[level].partition()
    return registered

cases = register_dimension(2, MESHES_2D)
cases_3d = register_dimension(3, MESHES_3D, prefix="3D", target_data='source_stretching_3d.data', nb_procs=NB_PROCS, partition=True)
case = cases["coarse"]
case_3d = cases_3d["coarse"]
run.printCases()
run.runCases()


## 2D checks


In [ ]:
t, b0 = read_son(case.dataFileName_, "SONDE_B0")
b0_0 = b0[0]
expected = b0_0 * np.exp(t)
err = np.max(np.abs(b0 - expected)) / abs(b0_0)
print(f"b0(0)={b0_0:.4f}  b0(t_end)={b0[-1]:.4f}  expected exp(t_end)={expected[-1]:.4f}")
print(f"max |b - B0 exp(t)| / B0 : {err:.3e}")
# the source must reproduce exp(t) to a few percent (first-order in dt
# for an explicit term; dt=2e-3 keeps the error well below 1%)
assert err < 0.01, f"induction source rate wrong: {err:.3e}"
print("OK: M1 source S=(b.grad)u has the correct magnitude and sign.")

import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
plt.plot(t, b0 / b0_0, label="b/B0 (simulated)")
plt.plot(t, np.exp(t), "k--", label="exp(t) (analytic)")
plt.xlabel("t [s]"); plt.ylabel("ratio"); plt.legend(); plt.grid(True); plt.show()


## 2D mesh convergence

The shared refinement ladder: `coarse` = the baseline mesh above (`17×17`),
`standard` = cells ×2 (`33×33`), `fine` = cells ×4 (`65×65`).

The growth $b(t) = B_0 e^{t}$ saturates at the **explicit-Euler temporal**
error of the lagged source (first order in $\Delta t$, common to every
mesh). To cancel it, the scan compares $b/b_0$ against the exact
explicit-Euler growth on the probe's own time grid,
$\mathrm{ref}(t_n) = \prod_j (1 + \Delta t_j)$. The residual is the
**spatial** error of the $(b\cdot\nabla)u$ reconstruction — which is exact
for the linear data velocity $u=(x,-y)$ at interior cells (Green–Gauss),
so the cancelled residual has **no spatial contribution**: its floor is the
level-independent time-scheme convergence slack ($\sim10^{-3}$ here). The
scan asserts a fixed cancelled bound at **every** level — a spatial error
would appear as growth under refinement, which the bound rules out.


In [ ]:
SONDE_TAG_SCAN = "SONDE_B0"

def read_son_dir(subdir, case_name, name):
    """(time, values) of probe `name` of case `case_name` in a build subdir."""
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def explicit_ref(t):
    """Exact explicit-Euler growth ratio on the probe time grid (lam = 1)."""
    t = np.asarray(t, dtype=float)
    ref = np.ones_like(t)
    for i in range(1, len(t)):
        ref[i:] *= 1.0 + (t[i] - t[i-1])
    return ref

def cancelled_error(subdir, case_name):
    """max |b(t)/b(0) - explicit-Euler growth| over all recorded times."""
    t, b = read_son_dir(subdir, case_name, SONDE_TAG_SCAN)
    if b[0] == 0:
        return 0.0
    return float(np.max(np.abs(b / b[0] - explicit_ref(t))))

scan = {"coarse": ("coarse", case.dataFileName_),
        "standard": (f"standard", cases["standard"].dataFileName_),
        "fine": (f"fine", cases["fine"].dataFileName_)}
scan_nodes = {"coarse": (17, 17), "standard": (33, 33), "fine": (65, 65)}
scan_err = {lvl: cancelled_error(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes     |ratio - explicit-Euler ref|")
for lvl in ("coarse", "standard", "fine"):
    nx, ny = scan_nodes[lvl]
    print(f"{lvl:<9} {nx:>3}x{ny:<4} {scan_err[lvl]:.3e}")
    # the residual floor is the implicit-iteration convergence slack
    # (level-independent); a spatial error would appear as growth
    # under refinement, asserted absent via the fixed bound.
    assert scan_err[lvl] < 5e-3, \
        f"cancelled stretching bound violated at {lvl}: {scan_err[lvl]:.3e}"
print("OK: cancelled bound holds at every level (grad(u) exact for the linear data velocity).")

## 3D checks


In [ ]:
t, b0 = read_son(case_3d.dataFileName_, "SONDE_B0")
b0_0 = b0[0]
expected = b0_0 * np.exp(t)
err = np.max(np.abs(b0 - expected)) / abs(b0_0)
print(f"b0(0)={b0_0:.4f}  b0(t_end)={b0[-1]:.4f}  expected exp(t_end)={expected[-1]:.4f}")
print(f"max |b - B0 exp(t)| / B0 : {err:.3e}")
# the source must reproduce exp(t) to a few percent (first-order in dt
# for an explicit term; dt=2e-3 keeps the error well below 1%)
assert err < 0.01, f"induction source rate wrong: {err:.3e}"
print("OK: M1 source S=(b.grad)u has the correct magnitude and sign.")

import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
plt.plot(t, b0 / b0_0, label="b/B0 (simulated)")
plt.plot(t, np.exp(t), "k--", label="exp(t) (analytic)")
plt.xlabel("t [s]"); plt.ylabel("ratio"); plt.legend(); plt.grid(True); plt.show()


## 3D mesh convergence

The shared refinement ladder: `coarse` = the baseline mesh above (`17×17×17`),
`standard` = cells ×2 (`33×33×33`), `fine` = cells ×4 (`65×65×65`).

The growth $b(t) = B_0 e^{t}$ saturates at the **explicit-Euler temporal**
error of the lagged source (first order in $\Delta t$, common to every
mesh). To cancel it, the scan compares $b/b_0$ against the exact
explicit-Euler growth on the probe's own time grid,
$\mathrm{ref}(t_n) = \prod_j (1 + \Delta t_j)$. The cancelled residual has
no spatial contribution to leading order — the Green–Gauss reconstruction
is exact for the linear data velocity at interior cells — so its floor is
the level-independent time-scheme convergence slack: the scan asserts a
fixed cancelled bound at **every** level (a spatial error would appear as
growth under refinement, which the bound rules out).

In [ ]:
SONDE_TAG_SCAN = "SONDE_B0"

def read_son_dir(subdir, case_name, name):
    """(time, values) of probe `name` of case `case_name` in a build subdir."""
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def explicit_ref(t):
    """Exact explicit-Euler growth ratio on the probe time grid (lam = 1)."""
    t = np.asarray(t, dtype=float)
    ref = np.ones_like(t)
    for i in range(1, len(t)):
        ref[i:] *= 1.0 + (t[i] - t[i-1])
    return ref

def cancelled_error(subdir, case_name):
    """max |b(t)/b(0) - explicit-Euler growth| over all recorded times."""
    t, b = read_son_dir(subdir, case_name, SONDE_TAG_SCAN)
    if b[0] == 0:
        return 0.0
    return float(np.max(np.abs(b / b[0] - explicit_ref(t))))

scan = {"coarse": ("3D/coarse", case_3d.dataFileName_),
        "standard": ("3D/standard", cases_3d["standard"].dataFileName_),
        "fine": ("3D/fine", cases_3d["fine"].dataFileName_)}
scan_nodes = {"coarse": (17, 17, 17), "standard": (33, 33, 33), "fine": (65, 65, 65)}
scan_err = {lvl: cancelled_error(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes     |ratio - explicit-Euler ref|")
for lvl in ("coarse", "standard", "fine"):
    nx, ny, nz = scan_nodes[lvl]
    print(f"{lvl:<9} {nx:>3}x{ny}x{nz:<3} {scan_err[lvl]:.3e}")
    # (level-independent); a spatial error would appear as growth
    # under refinement, asserted absent via the fixed bound.
    assert scan_err[lvl] < 5e-3, \
        f"cancelled stretching bound violated at {lvl}: {scan_err[lvl]:.3e}"
print("OK: cancelled bound holds at every level (grad(u) exact for the linear data velocity).")

## Computer Performance


In [ ]:
run.tablePerf()